# Detección de Fatiga (Quedarse Dormido) con YOLO Pose
Este script utiliza el modelo YOLO preentrenado con los puntos clave anatómicos del dataset COCO. 
Detecta si el usuario se está quedando dormido evaluando si la coordenada de la nariz cae por debajo del nivel promedio de los hombros.

In [1]:
import cv2
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [5]:
# Cargamos el modelo YOLO de estimación de posturas (COCO keypoints)
# No requiere entrenamiento, ya reconoce el esqueleto humano.
model = YOLO('yolov8n-pose.pt')

In [7]:
# Inicializamos la cámara web
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("No se pudo acceder a la cámara.")
        break

    # Inferencia para extraer los puntos clave del cuerpo
    resultados = model(frame, verbose=False)
    
    estado = "DESPIERTO / ATENTO"
    color = (0, 255, 0) # Verde

    # Verificamos si hay al menos una persona en pantalla
    if len(resultados[0].keypoints) > 0:
        puntos = resultados[0].keypoints.xy[0] 
        
        # Validamos que se detecten al menos los puntos de la cabeza y hombros (COCO tiene 17)
        if len(puntos) >= 7:
            nariz_y = float(puntos[0][1])        # Punto 0: Nariz
            hombro_izq_y = float(puntos[5][1])   # Punto 5: Hombro izquierdo
            hombro_der_y = float(puntos[6][1])   # Punto 6: Hombro derecho

            # Calculamos el nivel promedio de los hombros
            hombros_y_promedio = (hombro_izq_y + hombro_der_y) / 2

            # Lógica matemática: Si el valor Y de la nariz supera al de los hombros, la cabeza cayó
            if nariz_y > 0 and hombros_y_promedio > 0:
                if nariz_y > hombros_y_promedio:
                    estado = "ALERTA: FATIGA / DORMIDO"
                    color = (0, 0, 255) # Rojo

    # Dibujamos las conexiones y mostramos la alerta
    frame_dibujado = resultados[0].plot() 
    cv2.putText(frame_dibujado, estado, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)

    cv2.imshow('Monitor de Fatiga', frame_dibujado)
    
    # Detener con la tecla 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

[ WARN:0@200.193] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ WARN:0@200.193] global cap.cpp:435 open VIDEOIO(FFMPEG): raised OpenCV exception:

OpenCV(5.0.0) /io/opencv/modules/videoio/src/cap_ffmpeg_impl.hpp:1243: error: (-2:Unspecified error) in function 'bool CvCapture_FFMPEG::open(const char*, int, const cv::Ptr<cv::IStreamReader>&, const cv::VideoCaptureParameters&)'
> VIDEOIO/FFMPEG: Camera index out of range (expected: 'index < device_list->nb_devices'), where
>     'index' is 0
> must be less than
>     'device_list->nb_devices' is 0


[ERROR:0@200.193] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range


In [8]:
# Apagar cámara y cerrar ventanas
cap.release()
cv2.destroyAllWindows()